# 10 - Paraphraser Sweep

Re-paraphrase the same Qwen3-4B COTs using three additional paraphrasers:
Gemma-3-4B, Llama-3.2-3B, Mistral-7B. Run paraphrase_self for each.

If legibility scores are stable across paraphrasers, the metric is robust.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import gc, torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.data import load_gsm8k
from lib.paraphrase import paraphrase_cots
from lib.prefill import run_prefill_condition

dataset = load_gsm8k()

# Load original COTs
cots = []
for p in sorted(COT_CACHE.glob("*.json")):
    cots.append(json.loads(p.read_text()))
print(f"Loaded {len(cots)} COTs")

def paraphraser_tag(name):
    return name.split("/")[-1].lower().replace("-", "_").replace(".", "")

## Phase 1: Generate paraphrases with each paraphraser model

In [ ]:
for para_model in PARAPHRASER_SWEEP_MODELS:
    tag = paraphraser_tag(para_model)
    condition = f"parasweep_{tag}"

    n_done = len(list(PARAPHRASE_CACHE.glob(f"{condition}_*.json")))
    print(f"\n=== {para_model} ({tag}) === {n_done} already cached")

    if n_done >= len(cots):
        print("All done, skipping.")
        continue

    print(f"Loading {para_model}...")
    try:
        llm = LLM(model=para_model, dtype="bfloat16", max_model_len=4096)
        tokenizer = AutoTokenizer.from_pretrained(para_model)
    except Exception as e:
        print(f"Failed to load: {e}")
        continue

    paraphrase_cots(llm, tokenizer, cots, condition=condition, heavy=False)

    del llm, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{para_model} complete and unloaded.")

## Phase 2: Prefill PRIMARY_MODEL with each paraphraser's output

In [ ]:
print(f"Loading {PRIMARY_MODEL} for prefill...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)

for para_model in PARAPHRASER_SWEEP_MODELS:
    tag = paraphraser_tag(para_model)
    para_condition = f"parasweep_{tag}"
    prefill_condition = f"parasweep_self_{tag}"

    # Load paraphrases
    lookup = {}
    for p in PARAPHRASE_CACHE.glob(f"{para_condition}_*.json"):
        r = json.loads(p.read_text())
        lookup[r["problem_id"]] = r["paraphrased_cot"]
    print(f"\n{para_model}: {len(lookup)} paraphrases loaded")

    run_prefill_condition(llm, prefill_condition, PRIMARY_MODEL, dataset, lookup)

del llm
gc.collect()
torch.cuda.empty_cache()

## Analysis: Compare legibility across paraphrasers

In [ ]:
def get_acc(condition):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    return np.mean(results) if results else None

def bootstrap_ci(arr, n=10000, seed=42):
    rng = np.random.RandomState(seed)
    boots = sorted([rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)])
    return boots[int(0.025*n)], boots[int(0.975*n)]

no_cot_acc = get_acc("no_cot")
self_prefill_acc = get_acc("self_prefill")
total_value = self_prefill_acc - no_cot_acc

# Original paraphraser (Qwen3-8B)
orig_para_acc = get_acc("paraphrase_self")
orig_L = (orig_para_acc - no_cot_acc) / total_value if total_value > 0 else None

print(f"Baseline: no_cot={no_cot_acc:.3f}, self_prefill={self_prefill_acc:.3f}, total_value={total_value:.3f}")
print(f"Original paraphraser (Qwen3-8B): acc={orig_para_acc:.3f}, L={orig_L:.3f}")

sweep_results = [{"paraphraser": PARAPHRASER_MODEL, "acc": orig_para_acc, "L": orig_L}]

for para_model in PARAPHRASER_SWEEP_MODELS:
    tag = paraphraser_tag(para_model)
    acc = get_acc(f"parasweep_self_{tag}")
    L = (acc - no_cot_acc) / total_value if (acc is not None and total_value > 0) else None
    sweep_results.append({"paraphraser": para_model, "acc": acc, "L": L})
    print(f"{para_model}: acc={acc}, L={L}")

with open(RESULTS_DIR / "paraphraser_sweep.json", "w") as f:
    json.dump(sweep_results, f, indent=2, default=str)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

valid = [r for r in sweep_results if r["L"] is not None]
names = [r["paraphraser"].split("/")[-1] for r in valid]
l_vals = [r["L"] for r in valid]

colors = ["#8e44ad", "#27ae60", "#e67e22", "#2980b9"]
bars = ax.bar(range(len(valid)), l_vals, color=colors[:len(valid)],
              edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, l_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha="center", fontsize=10)

ax.set_xticks(range(len(valid)))
ax.set_xticklabels(names, rotation=15, fontsize=9)
ax.set_ylabel("Legibility score (L)")
ax.set_title("Legibility by Paraphraser Model", fontsize=13, fontweight="bold")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
ax.set_ylim(0, 1.5)

# Show variance
if len(l_vals) > 1:
    mean_L = np.mean(l_vals)
    std_L = np.std(l_vals)
    ax.axhline(y=mean_L, color="blue", linestyle="-", alpha=0.3,
               label=f"Mean L={mean_L:.3f} (std={std_L:.3f})")
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "paraphraser_sweep.png", dpi=200, bbox_inches="tight")
plt.show()